1. Baseline da classe majoritária → sempre prever "não vai converter".
2. Baseline da média (probabilidade média) → sempre prever a taxa histórica de conversão.

In [5]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# ==================================================
# Leitura dos dados processados
# ==================================================

input_path = Path("../data/processed/bank_marketing_processed.csv")
df = pd.read_csv(input_path)

print(f"Dataset carregado: {df.shape}")

# ==================================================
# Separação entre variáveis preditoras e alvo
# ==================================================

X = df.drop(columns=["y"])
y = df["y"]

# Caso a variável alvo ainda esteja como 'yes'/'no'
if y.dtype == "object":
    y = y.map({"no": 0, "yes": 1})
    
# ==================================================
# Divisão treino e teste
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape}")
print(f"Teste : {X_test.shape}")

Dataset carregado: (41188, 26)
Treino: (32950, 25)
Teste : (8238, 25)


In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ==========================================
# Escolha das variáveis que definem o perfil
# ==========================================

perfil = [
    "job",
    "education",
    "marital",
]

# ==========================================
# Junta X e y de treino
# ==========================================

train = X_train.copy()
train["y"] = y_train.values

# Classe majoritária global
classe_global = train["y"].mode()[0]

# Classe mais frequente para cada perfil
baseline_por_perfil = (
    train
    .groupby(perfil)["y"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

# ==========================================
# Faz a previsão
# ==========================================

def prever(row):
    chave = tuple(row[col] for col in perfil)
    return baseline_por_perfil.get(chave, classe_global)

y_pred = X_test.apply(prever, axis=1)

# ==========================================
# Avaliação
# ==========================================

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_pred):.4f}")

Accuracy : 0.8876
Precision: 0.7500
Recall   : 0.0032
F1-score : 0.0064
ROC AUC  : 0.5015


| Métrica   | Valor      | Interpretação                                                                                                                                                 |
| --------- | ---------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Accuracy  | **0,8876** | Muito alta, mas enganosa. O modelo acerta quase 89% dos clientes porque a maioria deles realmente não converte.                                               |
| Precision | **0,7500** | Quando o baseline prevê "Sim", ele acerta 75% das vezes. Parece bom, mas há um problema importante (o recall).                                                |
| Recall    | **0,0032** | Extremamente baixo. O modelo identifica apenas **0,32%** dos clientes que realmente aceitariam a oferta. Na prática, ele quase nunca prevê a classe positiva. |
| F1-score  | **0,0064** | Muito próximo de zero, indicando um desempenho muito ruim para identificar conversões.                                                                        |
| ROC AUC   | **0,5015** | Praticamente igual a um classificador aleatório (0,5). O modelo praticamente não consegue distinguir clientes que convertem dos que não convertem.            |
